In [1]:
from pyspark.sql.functions import (
    col, trim, upper, lower, initcap,
    when, lit, current_timestamp,
    to_date, regexp_replace, row_number,
    round as spark_round
)
from pyspark.sql.window import Window

# Read bronze_listings from Bronze Lakehouse
# Cross-Lakehouse read — same workspace, no credentials needed
df = spark.read.format("delta").table(
    "Blackwood_Bronze_Lakehouse.bronze_listings"
)

raw_count = df.count()
print(f"Records from bronze_listings: {raw_count}")
print("\nDistribution by source:")
df.groupBy("SourceRegion").count().show()
print("\nSchema:")
df.printSchema()


StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 3, Finished, Available, Finished, False)

Records from bronze_listings: 1800

Distribution by source:
+------------+-----+
|SourceRegion|count|
+------------+-----+
|      Europe|  600|
|         USA|  800|
|        APAC|  400|
+------------+-----+


Schema:
root
 |-- ListingID: string (nullable = true)
 |-- AgentName: string (nullable = true)
 |-- PropertyType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Bedrooms: integer (nullable = true)
 |-- Bathrooms: integer (nullable = true)
 |-- AreaSqFt: double (nullable = true)
 |-- ListingPrice: double (nullable = true)
 |-- ListDate: date (nullable = true)
 |-- Status: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- ListingURL: string (nullable = true)
 |-- SourceRegion: string (nullable = true)
 |-- SourceFile: string (nullable = true)
 |-- _ingestion_ts: timestamp (nullable = true)
 |-- PricePerSqFt: double (nullable = true)
 |-- ViewType: string (nullable = 

In [2]:
# MANDATORY FIELDS — a row without these cannot be used in any analysis
# ListingID — cannot deduplicate or track without a primary key
# ListingPrice — the fundamental measure of every listing
# City — cannot do geographic analysis without city
# ListDate — cannot do time-based analysis without a date

df_clean = df

before = df_clean.count()
df_clean = df_clean.filter(col("ListingID").isNotNull())
print(f"Rule 1 — Null ListingID:    {before - df_clean.count()} dropped")

before = df_clean.count()
df_clean = df_clean.filter(
    col("ListingPrice").isNotNull() &
    (col("ListingPrice") > 0)
)
print(f"Rule 2 — Invalid price:     {before - df_clean.count()} dropped")

before = df_clean.count()
df_clean = df_clean.filter(col("City").isNotNull())
print(f"Rule 3 — Null City:         {before - df_clean.count()} dropped")

before = df_clean.count()
df_clean = df_clean.filter(col("ListDate").isNotNull())
print(f"Rule 4 — Null ListDate:     {before - df_clean.count()} dropped")

StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 4, Finished, Available, Finished, False)

Rule 1 — Null ListingID:    0 dropped
Rule 2 — Invalid price:     0 dropped
Rule 3 — Null City:         0 dropped
Rule 4 — Null ListDate:     0 dropped


In [3]:
# Bedrooms and bathrooms must be positive integers
# Zero bedrooms is valid — studio apartments
# Negative values are data errors

before = df_clean.count()
df_clean = df_clean.filter(
    col("Bedrooms").isNull() |
    (col("Bedrooms") >= 0)
)
print(f"Rule 5 — Invalid bedrooms:  {before - df_clean.count()} dropped")

before = df_clean.count()
df_clean = df_clean.filter(
    col("Bathrooms").isNull() |
    (col("Bathrooms") >= 0)
)
print(f"Rule 6 — Invalid bathrooms: {before - df_clean.count()} dropped")

# Valid area range — 50 sqft minimum (very small studio),
# 50,000 sqft maximum (very large estate)
before = df_clean.count()
df_clean = df_clean.filter(
    col("AreaSqFt").isNull() |
    ((col("AreaSqFt") >= 50) & (col("AreaSqFt") <= 50000))
)
print(f"Rule 7 — Invalid area:      {before - df_clean.count()} dropped")

StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 5, Finished, Available, Finished, False)

Rule 5 — Invalid bedrooms:  0 dropped
Rule 6 — Invalid bathrooms: 0 dropped
Rule 7 — Invalid area:      0 dropped


In [4]:
# WHY THIS MATTERS:
# USA has "Single Family" and "Condo"
# Europe has "Flat" and "Bungalow"
# APAC has "Apartment" and "Studio"
# None of these map to each other without standardisation
#
# UNIFIED PROPERTY TYPE MAPPING:
# Apartment / Flat / Studio / Condo   → Apartment
# House / Single Family / Bungalow    → House
# Villa                               → Villa
# Townhouse                           → Townhouse
# Penthouse                           → Penthouse
# Multi-Family                        → Multi-Family
# everything else                     → Other

df_clean = df_clean.withColumn("PropertyType",
    when(upper(col("PropertyType")).isin(
        "APARTMENT", "FLAT", "STUDIO", "CONDO"), lit("Apartment"))
   .when(upper(col("PropertyType")).isin(
        "HOUSE", "SINGLE FAMILY", "BUNGALOW"), lit("House"))
   .when(upper(col("PropertyType")) == "VILLA",      lit("Villa"))
   .when(upper(col("PropertyType")) == "TOWNHOUSE",  lit("Townhouse"))
   .when(upper(col("PropertyType")) == "PENTHOUSE",  lit("Penthouse"))
   .when(upper(col("PropertyType")) == "MULTI-FAMILY", lit("Multi-Family"))
   .otherwise(lit("Other")))

print("Property type distribution after standardisation:")
df_clean.groupBy("PropertyType").count() \
    .orderBy("count", ascending=False).show()

StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 6, Finished, Available, Finished, False)

Property type distribution after standardisation:
+------------+-----+
|PropertyType|count|
+------------+-----+
|       House|  428|
|   Apartment|  353|
|   Penthouse|  303|
|   Townhouse|  296|
|       Villa|  289|
|Multi-Family|  131|
+------------+-----+



In [5]:
# Final country name standardisation
# Handles any remaining inconsistencies from the 3 source files

df_clean = df_clean \
    .withColumn("Country", initcap(trim(col("Country")))) \
    .withColumn("City",    initcap(trim(col("City"))))

print("Top 10 countries after standardisation:")
df_clean.groupBy("Country").count() \
    .orderBy("count", ascending=False).show(10, truncate=False)

StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 7, Finished, Available, Finished, False)

Top 10 countries after standardisation:
+--------------+-----+
|Country       |count|
+--------------+-----+
|Usa           |607  |
|Australia     |247  |
|Canada        |193  |
|France        |147  |
|United Kingdom|142  |
|Germany       |125  |
|Hong Kong     |98   |
|Spain         |86   |
|Singapore     |55   |
|Netherlands   |52   |
+--------------+-----+
only showing top 10 rows



In [6]:
# WHY DEDUPLICATION IS NEEDED ACROSS REGIONS:
# The pipeline might be rerun — dedup prevents double counting
# Also if the same property is somehow listed in two regional files
# (edge case but possible) we keep only the latest version
#
# WHY WINDOW FUNCTION NOT DISTINCT:
# DISTINCT only works when all columns are identical
# Window row_number keeps one row per ListingID
# prioritising the most recently ingested record

window_spec = Window.partitionBy("ListingID") \
    .orderBy(col("_ingestion_ts").desc())

before = df_clean.count()
df_clean = df_clean \
    .withColumn("_row_num", row_number().over(window_spec)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num")

print(f"Rule 10 — Duplicates removed: {before - df_clean.count()} rows")
print(f"\nFinal clean count: {df_clean.count()} of {raw_count}")

StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 8, Finished, Available, Finished, False)

Rule 10 — Duplicates removed: 0 rows

Final clean count: 1800 of 1800


In [7]:
# Add Silver audit columns and drop Bronze-specific ones
df_silver = df_clean \
    .withColumn("_silver_timestamp", current_timestamp()) \
    .withColumn("_quality_passed",   lit(True)) \
    .drop("_ingestion_ts")  # replaced by _silver_timestamp

print(f"Silver columns: {len(df_silver.columns)}")
print("Final schema:")
df_silver.printSchema()

StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 9, Finished, Available, Finished, False)

Silver columns: 21
Final schema:
root
 |-- ListingID: string (nullable = true)
 |-- AgentName: string (nullable = true)
 |-- PropertyType: string (nullable = false)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Bedrooms: integer (nullable = true)
 |-- Bathrooms: integer (nullable = true)
 |-- AreaSqFt: double (nullable = true)
 |-- ListingPrice: double (nullable = true)
 |-- ListDate: date (nullable = true)
 |-- Status: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- ListingURL: string (nullable = true)
 |-- SourceRegion: string (nullable = true)
 |-- SourceFile: string (nullable = true)
 |-- PricePerSqFt: double (nullable = true)
 |-- ViewType: string (nullable = true)
 |-- FurnishedStatus: string (nullable = true)
 |-- _silver_timestamp: timestamp (nullable = false)
 |-- _quality_passed: boolean (nullable = false)



In [8]:
# Full overwrite — Silver always rebuilt from Bronze
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Blackwood_SilverLakehouse.silver_listings")

total = spark.table("Blackwood_SilverLakehouse.silver_listings").count()
print(f"silver_listings written: {total} rows")

StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 10, Finished, Available, Finished, False)

silver_listings written: 1800 rows


In [9]:
# OPTIMIZE + ZORDER on the most common filter patterns
# Real estate dashboards always filter by Country and Status
# ZORDERing on these columns makes those queries dramatically faster

spark.sql("""
    OPTIMIZE Blackwood_SilverLakehouse.silver_listings
    ZORDER BY (Country, Status, PropertyType)
""")
print("OPTIMIZE complete ✅")

# spark.sql("DESCRIBE DETAIL silver_listings") \
#     .select("numFiles", "sizeInBytes", "numRows") \
#     .show(truncate=False)

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "Blackwood_SilverLakehouse.silver_listings")
rows = spark.table("Blackwood_SilverLakehouse.silver_listings").count()
detail_df = delta_table.detail().select("numFiles", "sizeInBytes")

# Display the results
from pyspark.sql.functions import lit
detail_df.withColumn("numRows", lit(rows)).show(truncate=False)

StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 11, Finished, Available, Finished, False)

OPTIMIZE complete ✅
+--------+-----------+-------+
|numFiles|sizeInBytes|numRows|
+--------+-----------+-------+
|1       |70044      |1800   |
+--------+-----------+-------+



In [10]:
print("=== Silver Listings — Final Verification ===\n")

# Row counts by region
print("Rows by source region:")
spark.sql("""
    SELECT SourceRegion, COUNT(*) AS Rows
    FROM   Blackwood_SilverLakehouse.silver_listings
    GROUP BY SourceRegion
    ORDER BY Rows DESC
""").show()

# Global property type breakdown
print("Property type breakdown — unified across all regions:")
spark.sql("""
    SELECT PropertyType, COUNT(*) AS Listings,
           ROUND(AVG(ListingPrice), 0) AS AvgPrice
    FROM   Blackwood_SilverLakehouse.silver_listings
    GROUP BY PropertyType
    ORDER BY Listings DESC
""").show()

# Status distribution
print("Status distribution:")
spark.sql("""
    SELECT Status, COUNT(*) AS Count
    FROM   Blackwood_SilverLakehouse.silver_listings
    GROUP BY Status
    ORDER BY Count DESC
""").show()

# Price comparison across regions
print("Average listing price by region:")
spark.sql("""
    SELECT SourceRegion,
           COUNT(*) AS Listings,
           ROUND(MIN(ListingPrice), 0) AS MinPrice,
           ROUND(MAX(ListingPrice), 0) AS MaxPrice,
           ROUND(AVG(ListingPrice), 0) AS AvgPrice
    FROM   Blackwood_SilverLakehouse.silver_listings
    GROUP BY SourceRegion
    ORDER BY AvgPrice DESC
""").show()

# Top 10 cities globally
print("Top 10 cities by listing count:")
spark.sql("""
    SELECT City, Country, COUNT(*) AS Listings
    FROM   Blackwood_SilverLakehouse.silver_listings
    GROUP BY City, Country
    ORDER BY Listings DESC
    LIMIT 10
""").show(truncate=False)

# APAC-specific columns populated correctly
print("APAC extra columns (should be null for USA + Europe):")
spark.sql("""
    SELECT SourceRegion,
           COUNT(ViewType) AS HasViewType,
           COUNT(FurnishedStatus) AS HasFurnished
    FROM Blackwood_SilverLakehouse.silver_listings
    GROUP BY SourceRegion
""").show()

StatementMeta(, 51182c88-c0ec-4060-a271-e08b681b9de9, 12, Finished, Available, Finished, False)

=== Silver Listings — Final Verification ===

Rows by source region:
+------------+----+
|SourceRegion|Rows|
+------------+----+
|         USA| 800|
|      Europe| 600|
|        APAC| 400|
+------------+----+

Property type breakdown — unified across all regions:
+------------+--------+---------+
|PropertyType|Listings| AvgPrice|
+------------+--------+---------+
|       House|     428|3291731.0|
|   Apartment|     353|3214980.0|
|   Penthouse|     303|3073766.0|
|   Townhouse|     296|3034527.0|
|       Villa|     289|3019024.0|
|Multi-Family|     131|2278878.0|
+------------+--------+---------+

Status distribution:
+------+-----+
|Status|Count|
+------+-----+
|Active| 1129|
|  Sold|  589|
| Other|   82|
+------+-----+

Average listing price by region:
+------------+--------+--------+---------+---------+
|SourceRegion|Listings|MinPrice| MaxPrice| AvgPrice|
+------------+--------+--------+---------+---------+
|      Europe|     600|200000.0|7988000.0|4028243.0|
|        APAC|     400|